# Mini Project - GrowthRetail Hypothesis-Testing Analysis

**Role:** Data analyst at **GrowthRetail**.

**Problem statement:** GrowthRetail recently introduced two changes - a new
loyalty programme and a regional product strategy. Management requires
**statistical validation** that these changes produced genuine effects *before*
they are scaled. This notebook conducts a complete hypothesis-testing analysis
across three business scenarios.

For **each** scenario we:
1. Formulate the hypotheses (null and alternative).
2. Select and **justify** the appropriate test (by data type and comparison).
3. Perform the test in Python using SciPy.
4. Interpret the p-value against a significance level of **alpha = 0.05**.
5. Make a decision and write an evidence-based business recommendation.

| # | Scenario | Comparison | Test |
|---|----------|-----------|------|
| 1 | Loyalty programme | Same customers, before vs after (paired) | Paired t-test |
| 2 | Two-store sales | Two separate stores (independent) | Independent t-test |
| 3 | Region vs preference | Two categorical variables | Chi-square test |

## Setup

Import the libraries and define a small helper that applies the decision rule
(`p < alpha` -> reject H0) consistently across all three scenarios.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency

ALPHA = 0.05

def decide(p, alpha=ALPHA):
    """Return the formal decision and the significance verdict for a p-value."""
    if p < alpha:
        return "Reject H0", "Significant"
    return "Fail to reject H0", "Not significant"

print("Setup complete. Significance level alpha =", ALPHA)

Setup complete. Significance level alpha = 0.05


---
## Scenario 1 - Loyalty Programme (paired comparison)

**Business question:** Did enrolling customers in the loyalty programme
*significantly increase* their monthly spending?

**Data:** the **same** customers' monthly spending, recorded **before** and
**after** enrolment (`data/loyalty_spending.csv`).

**Hypotheses**
- H0 (null): mean spending after = mean spending before (no change).
- H1 (alternative): mean spending after ≠ mean spending before (a change).

**Test choice and justification:** the two measurements come from the **same**
customers, so each "after" value is paired with that customer's own "before"
value. The correct test is the **paired t-test** (`stats.ttest_rel`). Pairing
removes customer-to-customer variation and isolates the programme's effect.

In [2]:
loyalty = pd.read_csv("data/loyalty_spending.csv")
print(loyalty.head())
print("Customers:", len(loyalty))

before = loyalty["before"]
after = loyalty["after"]

t_stat, p_value = stats.ttest_rel(before, after)
decision, verdict = decide(p_value)

print()
print("Mean spending before:", round(before.mean(), 1))
print("Mean spending after :", round(after.mean(), 1))
print("Mean increase       :", round(after.mean() - before.mean(), 1))
print("t-statistic:", round(t_stat, 4))
print("p-value:", p_value)
print("Decision:", decision, "->", verdict)

  customer_id  before  after
0         C01    2500   2900
1         C02    2800   3200
2         C03    2300   2600
3         C04    3100   3500
4         C05    2900   3300
Customers: 12

Mean spending before: 2741.7
Mean spending after : 3108.3
Mean increase       : 366.7
t-statistic: -32.6313
p-value: 2.6693797337801966e-12
Decision: Reject H0 -> Significant


**Interpretation and recommendation (Scenario 1):** the p-value is far below
0.05, so we **reject H0**. Average monthly spending rose from about ₹2,742 to
about ₹3,108 - an increase of roughly ₹367 per customer - and this increase is
very unlikely to be chance.

> **Recommendation:** the loyalty programme genuinely increased spending. Expand
> it to more customers, while confirming the extra revenue exceeds the cost of
> running the programme.

---
## Scenario 2 - Two-Store Comparison (independent comparison)

**Business question:** Do the two stores have *significantly different* average
daily sales?

**Data:** daily sales recorded for **two different stores** over the same period
(`data/store_daily_sales.csv`).

**Hypotheses**
- H0 (null): mean daily sales of Store A = mean daily sales of Store B.
- H1 (alternative): the two stores' mean daily sales differ.

**Test choice and justification:** Store A and Store B are **two separate,
unrelated groups** of days - there is no natural pairing between a day at Store A
and a day at Store B. The correct test is the **independent (two-sample) t-test**
(`stats.ttest_ind`).

In [3]:
stores = pd.read_csv("data/store_daily_sales.csv")
print(stores.head())
print("Days recorded:", len(stores))

store_a = stores["store_a"]
store_b = stores["store_b"]

t_stat, p_value = stats.ttest_ind(store_a, store_b)
decision, verdict = decide(p_value)

print()
print("Store A mean daily sales:", round(store_a.mean(), 1))
print("Store B mean daily sales:", round(store_b.mean(), 1))
print("t-statistic:", round(t_stat, 4))
print("p-value:", p_value)
print("Decision:", decision, "->", verdict)

   day  store_a  store_b
0    1     1520     1350
1    2     1480     1400
2    3     1550     1320
3    4     1600     1380
4    5     1490     1410
Days recorded: 14

Store A mean daily sales: 1533.2
Store B mean daily sales: 1366.1
t-statistic: 13.7001
p-value: 2.1071166326955632e-13
Decision: Reject H0 -> Significant


**Interpretation and recommendation (Scenario 2):** the p-value is far below
0.05, so we **reject H0**. Store A averages about ₹1,533 in daily sales versus
about ₹1,366 for Store B, and this gap is statistically significant rather than
day-to-day noise.

> **Recommendation:** the two stores genuinely differ. Investigate what drives
> Store A's stronger performance (location, staff, product mix, promotions) and
> replicate those practices at Store B. Note that significance confirms the
> difference is real, but the business should also weigh whether the size of the
> gap justifies the cost of intervention.

---
## Scenario 3 - Region and Preference (categorical relationship)

**Business question:** Is a customer's **region** related to their **preferred
product category**?

**Data:** a survey crosstab of region (North, South, East) against preferred
category (Electronics, Clothing, Groceries), stored as a contingency table
(`data/region_preference.csv`).

**Hypotheses**
- H0 (null): region and preferred category are independent (unrelated).
- H1 (alternative): region and preferred category are associated (related).

**Test choice and justification:** both variables are **categorical**, and we are
asking whether they are associated. The correct test is the **chi-square test of
independence** (`chi2_contingency`) applied to the contingency table.

In [4]:
pref = pd.read_csv("data/region_preference.csv", index_col="region")
print(pref)

observed = pref.values
chi2, p_value, dof, expected = chi2_contingency(observed)
decision, verdict = decide(p_value)

print()
print("Chi-square statistic:", round(chi2, 4))
print("Degrees of freedom:", dof)
print("p-value:", p_value)
print("Decision:", decision, "->", verdict)
print()
print("Expected counts under independence:")
print(pd.DataFrame(expected.round(1), index=pref.index, columns=pref.columns))

        Electronics  Clothing  Groceries
region                                  
North            60        25         15
South            20        55         25
East             30        30         40

Chi-square statistic: 49.6023
Degrees of freedom: 4
p-value: 4.371617606844666e-10
Decision: Reject H0 -> Significant

Expected counts under independence:
        Electronics  Clothing  Groceries
region                                  
North          36.7      36.7       26.7
South          36.7      36.7       26.7
East           36.7      36.7       26.7


**Interpretation and recommendation (Scenario 3):** the p-value is far below
0.05, so we **reject H0**. Region and product preference are significantly
associated: the North leans toward Electronics, the South toward Clothing, and
the East toward Groceries. Comparing observed with expected counts shows these
regional skews are larger than independence would predict.

> **Recommendation:** product preference depends on region. Adopt a
> **region-specific strategy** - tailor stocking, assortment, and marketing to
> each region's leading category instead of using one national plan.

---
## Summary of results

In [5]:
# Re-run all three tests and assemble a single summary table.
loyalty = pd.read_csv("data/loyalty_spending.csv")
stores = pd.read_csv("data/store_daily_sales.csv")
pref = pd.read_csv("data/region_preference.csv", index_col="region")

_, p1 = stats.ttest_rel(loyalty["before"], loyalty["after"])
_, p2 = stats.ttest_ind(stores["store_a"], stores["store_b"])
_, p3, _, _ = chi2_contingency(pref.values)

summary = pd.DataFrame({
    "Scenario": ["1. Loyalty programme", "2. Two-store sales", "3. Region vs preference"],
    "Test": ["Paired t-test", "Independent t-test", "Chi-square test"],
    "p-value": [p1, p2, p3],
    "Decision": [decide(p)[0] for p in (p1, p2, p3)],
    "Verdict": [decide(p)[1] for p in (p1, p2, p3)],
})
summary

,Scenario,Test,p-value,Decision,Verdict
0,1. Loyalty programme,Paired t-test,2.669380e-12,Reject H0,Significant
1,2. Two-store sales,Independent t-test,2.107117e-13,Reject H0,Significant
2,3. Region vs preference,Chi-square test,4.371618e-10,Reject H0,Significant


## Conclusion

All three changes show **statistically significant** effects, so management has
evidence-based grounds to act:

1. **Loyalty programme** significantly increased spending -> **expand it** (after
   a cost check).
2. **The two stores** differ significantly -> **investigate and replicate** Store
   A's practices.
3. **Region and preference** are significantly related -> **adopt a regional
   product strategy**.

**Caveats (honest reporting):**
- Significance confirms an effect is *real*; it does not measure how *large* or
  *commercially worthwhile* it is. Always pair the p-value with the effect size
  and a cost/benefit view.
- These conclusions rest on the sample data provided; on real data, also check
  sample size and basic test assumptions, and distinguish correlation from
  causation before claiming a change *caused* the effect.